# 06. Extended calm-day subsample (Table A.17, referee Q7)

Tests whether the cross-resolution dissonance is an artefact of the
geopolitical-stress windows. Procedure:

1. Fit GMM regimes per frequency on the *full* sample (boundary frozen).
2. Compute daily realised volatility from 5m returns.
3. Define the calm-day set as days with daily RV below the median,
   additionally excluding the peak-stress event window.
4. Restrict the 5m index to calm days and recompute the ARI matrix.

We run the full procedure on `CL` and display per-pair ARIs.

**Re-run command**: `python run.py extended_calm_subsample`

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

PROJECT = Path.cwd()
_nb_file = globals().get("__vsc_ipynb_file__")
if _nb_file is not None:
    PROJECT = Path(_nb_file).resolve().parent.parent
elif PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

OUT = PROJECT / "outputs"
OUT_2022 = PROJECT / "outputs_2022"
DATA = PROJECT / "data"
DATA_2022 = PROJECT / "data_2022"

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)

## Step 1 -- load raw 5m OHLC for `CL`

Calls `src.data.data_ib.load_5m_ohlc` directly on `data/CL_5m.csv`. This is the same loader the pipeline uses; it parses mixed-offset DST timestamps as UTC then converts to NY local time.

In [2]:
from src.data.data_ib import load_5m_ohlc

FOCAL = "CL"
df_5m = load_5m_ohlc(DATA / f"{FOCAL}_5m.csv")
print(f"{FOCAL}_5m: {len(df_5m):,} bars, "
      f"{df_5m.index.min()} -> {df_5m.index.max()}, "
      f"{df_5m.index.normalize().nunique()} trading days")
df_5m.head()

CL_5m: 34,896 bars, 2025-11-02 18:00:00-05:00 -> 2026-05-01 16:55:00-04:00, 155 trading days


,Open,High,Low,Close,Volume
Date,,,,,
2025-11-02 18:00:00-05:00,58.36,58.43,58.10,58.29,2872.0
2025-11-02 18:05:00-05:00,58.30,58.37,58.29,58.35,669.0
2025-11-02 18:10:00-05:00,58.35,58.36,58.29,58.30,367.0
2025-11-02 18:15:00-05:00,58.30,58.33,58.26,58.33,394.0
2025-11-02 18:20:00-05:00,58.33,58.33,58.30,58.31,313.0


## Step 2 -- run the calm-day subsample analysis on the focal asset

In [3]:
from src.experiments.exp_05_calm_subsample import calm_day_subsample_ari
from src.core.config import EPISODES

event_2026, _ = EPISODES["2026_iran"]
res = calm_day_subsample_ari(
    df_5m, FOCAL, exclude_window=event_2026, quantile=0.5,
)
pd.DataFrame([res])

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Win

,symbol,quantile,calm_n_days,total_n_days,rv_cutoff,rv_median,full_sample_ari,calm_subsample_ari,ari_5m_15m,ari_15m_1h,ari_1h_1d
0,CL,0.5,77,155,0.018,0.018,0.2,0.074,0.338,0.063,8.330e-04


## Step 3 -- compare full-sample vs calm-subsample mean ARI

The "structural" prediction (paper's claim) is that the calm-subsample
mean ARI is close to the full-sample value, demonstrating the dissonance
is a property of the cross-frequency decomposition rather than the
stress episode.

In [4]:
delta = res["calm_subsample_ari"] - res["full_sample_ari"]
display(Markdown(
    f"- full-sample mean ARI    : **{res['full_sample_ari']:.4f}**\n"
    f"- calm-subsample mean ARI : **{res['calm_subsample_ari']:.4f}**\n"
    f"- delta (calm - full)     : **{delta:+.4f}**\n"
    f"- calm-day count          : **{res['calm_n_days']}** of **{res['total_n_days']}** total"
))

- full-sample mean ARI    : **0.2002**
- calm-subsample mean ARI : **0.0742**
- delta (calm - full)     : **-0.1259**
- calm-day count          : **77** of **155** total

## Cached all-asset summary -- `outputs/calm_day_subsample_ari.csv`

In [5]:
p = OUT / "calm_day_subsample_ari.csv"
display(pd.read_csv(p).round(3) if p.exists() else Markdown(f"`{p}` missing"))

,symbol,quantile,calm_n_days,total_n_days,rv_cutoff,rv_median,full_sample_ari,calm_subsample_ari,ari_5m_15m,ari_15m_1h,ari_1h_1d
0,SPY,0.5,54,124,0.009,0.009,0.178,0.109,0.460,0.049,0.067
1,USDJPY,0.5,54,156,0.004,0.004,0.104,0.050,0.256,0.050,0.007
2,CL,0.5,77,155,0.018,0.018,0.200,0.074,0.338,0.063,0.001
3,GLD,0.5,52,124,0.014,0.014,0.144,0.108,0.555,0.017,0.022
